In [ ]:
import os
import re
import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from astropy.timeseries import BoxLeastSquares

In [ ]:
BASE_PATH = (
    "/home/luciam/Documents/starry/"
    "Orion_yceb/dat_orion_sergio/sample_3/"
    "831359(1)/Light_Curve_Files/"
)

OUTPUT_DIR = "block1_orbital"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

In [ ]:
files = sorted(
    glob.glob(
        os.path.join(
            BASE_PATH,
            "*.txt"
        )
    )
)

print(
    f"Found {len(files)} files"
)

In [ ]:
tic_files = {}

for f in files:

    fname = os.path.basename(f)

    m = re.search(
        r'tic\s+(\d+)_S_(\d+)_LC',
        fname
    )

    if m is None:
        continue

    tic = int(
        m.group(1)
    )

    sector = int(
        m.group(2)
    )

    tic_files.setdefault(
        tic,
        []
    )

    tic_files[tic].append(
        (sector, f)
    )

print(
    f"Found {len(tic_files)} TICs"
)

In [ ]:
def load_sector_file(fname):

    df = pd.read_csv(
        fname,
        delim_whitespace=True,
        comment="#",
        header=None
    )

    arr = df.values

    time_col = None
    flux_col = None

    for i in range(arr.shape[1]):

        med = np.nanmedian(
            arr[:,i]
        )

        if (
            time_col is None
            and
            1000 < med < 5000
        ):
            time_col = i

        if (
            flux_col is None
            and
            0.5 < med < 1.5
        ):
            flux_col = i

    if time_col is None:
        raise ValueError(
            f"No TIME column in {fname}"
        )

    if flux_col is None:
        raise ValueError(
            f"No FLUX column in {fname}"
        )

    time = arr[:,time_col]
    flux = arr[:,flux_col]

    mask = (
        np.isfinite(time)
        &
        np.isfinite(flux)
    )

    time = time[mask]
    flux = flux[mask]

    srt = np.argsort(time)

    return (
        time[srt],
        flux[srt]
    )

In [ ]:
def build_global_lc(
    sectors
):

    all_time = []
    all_flux = []

    for sector, fname in sectors:

        t, f = load_sector_file(
            fname
        )

        all_time.append(t)
        all_flux.append(f)

    time = np.concatenate(
        all_time
    )

    flux = np.concatenate(
        all_flux
    )

    srt = np.argsort(time)

    return (
        time[srt],
        flux[srt]
    )

In [ ]:
def get_bls_period(
    time,
    flux,
    min_period=0.5,
    max_period=20
):

    flux = flux / np.nanmedian(flux)

    periods = np.linspace(
        min_period,
        max_period,
        20000
    )

    durations = np.linspace(
        0.02,
        0.20,
        25
    )

    bls = BoxLeastSquares(
        time,
        flux
    )

    result = bls.power(
        periods,
        durations
    )

    best = np.argmax(
        result.power
    )

    return (
        result.period[best],
        result.transit_time[best],
        result.depth[best],
        result.duration[best],
        result
    )

In [ ]:
def phase_plot(
    ax,
    time,
    flux,
    period,
    title
):

    flux = (
        flux /
        np.nanmedian(flux)
    )

    phase = (
        (time % period)
        / period
    )

    ax.scatter(
        phase,
        flux,
        s=1,
        alpha=0.10
    )

    bins = np.linspace(
        0,
        1,
        120
    )

    centers = 0.5 * (
        bins[:-1]
        + bins[1:]
    )

    medians = []

    for i in range(
        len(bins)-1
    ):

        mask = (
            (phase >= bins[i])
            &
            (phase < bins[i+1])
        )

        if np.sum(mask) > 5:

            medians.append(
                np.nanmedian(
                    flux[mask]
                )
            )

        else:

            medians.append(
                np.nan
            )

    ax.plot(
        centers,
        medians,
        lw=3
    )

    ax.set_xlim(
        0,
        1
    )

    ax.set_title(
        title
    )

    ax.set_xlabel(
        "Phase"
    )

In [ ]:
def save_phase_diagnostic(
    time,
    flux,
    P,
    outfile
):

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(15,4)
    )

    phase_plot(
        axes[0],
        time,
        flux,
        P/2,
        f"P/2 = {P/2:.4f} d"
    )

    phase_plot(
        axes[1],
        time,
        flux,
        P,
        f"P = {P:.4f} d"
    )

    phase_plot(
        axes[2],
        time,
        flux,
        2*P,
        f"2P = {2*P:.4f} d"
    )

    axes[0].set_ylabel(
        "Normalized Flux"
    )

    plt.tight_layout()

    plt.savefig(
        outfile,
        dpi=300
    )

    plt.close()

In [ ]:
summary = []

for tic in sorted(
    tic_files.keys()
):

    print(
        "\n====================="
    )

    print(
        f"TIC {tic}"
    )

    try:

        sectors = sorted(
            tic_files[tic]
        )

        time_all, flux_all = (
            build_global_lc(
                sectors
            )
        )

        (
            P_bls,
            t0,
            depth,
            duration,
            result
        ) = get_bls_period(
            time_all,
            flux_all
        )

        print(
            f"P_BLS = {P_bls:.5f}"
        )

        summary.append({

            "tic": tic,

            "P_bls": P_bls,

            "depth": depth,

            "duration": duration,

            "n_points":
            len(time_all)

        })

        # ------------------------
        # BLS POWER
        # ------------------------

        plt.figure(
            figsize=(7,4)
        )

        plt.plot(
            result.period,
            result.power,
            color='k'
        )

        plt.axvline(
            P_bls,
            color='red',
            lw=2
        )

        plt.xlabel(
            "Period [d]"
        )

        plt.ylabel(
            "BLS Power"
        )

        plt.title(
            f"TIC {tic}"
        )

        plt.tight_layout()

        plt.savefig(
            os.path.join(
                OUTPUT_DIR,
                f"TIC_{tic}_BLS.png"
            ),
            dpi=300
        )

        plt.close()

        # ------------------------
        # PHASE CHECK
        # ------------------------

        save_phase_diagnostic(
            time_all,
            flux_all,
            P_bls,
            os.path.join(
                OUTPUT_DIR,
                f"TIC_{tic}_phasecheck.png"
            )
        )

    except Exception as e:

        print(
            f"ERROR: {e}"
        )

In [ ]:
summary = pd.DataFrame(
    summary
)

summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "orbital_candidates.csv"
    ),
    index=False
)

print(
    "\nDONE"
)

print(
    summary
)

In [ ]:
# ============================================================
# BLOCK 1 v4
# GLOBAL ORBITAL PERIOD ANALYSIS
# ============================================================

import os
import re
import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from astropy.timeseries import BoxLeastSquares

# ============================================================
# CONFIG
# ============================================================

BASE_PATH = (
    "/home/luciam/Documents/starry/"
    "Orion_yceb/dat_orion_sergio/sample_3/"
    "831359(1)/Light_Curve_Files/"
)

OUTPUT_DIR = "block1_orbital"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

# ============================================================
# FIND FILES
# ============================================================

files = sorted(
    glob.glob(
        os.path.join(
            BASE_PATH,
            "*.txt"
        )
    )
)

print(
    f"\nFound {len(files)} files"
)

# ============================================================
# GROUP FILES BY TIC
# ============================================================

tic_files = {}

for f in files:

    fname = os.path.basename(f)

    match = re.search(
        r'tic\s+(\d+)_S_(\d+)_LC',
        fname
    )

    if match is None:
        continue

    tic = int(match.group(1))
    sector = int(match.group(2))

    tic_files.setdefault(
        tic,
        []
    )

    tic_files[tic].append(
        (sector, f)
    )

print(
    f"Found {len(tic_files)} TICs"
)

# ============================================================
# LOAD LIGHT CURVE
# ============================================================

def load_sector_file(fname):

    df = pd.read_csv(
        fname,
        delim_whitespace=True,
        comment="#",
        header=None
    )

    arr = df.values

    time_col = None
    flux_col = None

    for i in range(arr.shape[1]):

        med = np.nanmedian(
            arr[:, i]
        )

        if (
            time_col is None
            and
            1000 < med < 5000
        ):
            time_col = i

        if (
            flux_col is None
            and
            0.5 < med < 1.5
        ):
            flux_col = i

    if time_col is None:
        raise ValueError(
            f"TIME column not found in {fname}"
        )

    if flux_col is None:
        raise ValueError(
            f"FLUX column not found in {fname}"
        )

    time = arr[:, time_col]
    flux = arr[:, flux_col]

    mask = (
        np.isfinite(time)
        &
        np.isfinite(flux)
    )

    time = time[mask]
    flux = flux[mask]

    srt = np.argsort(time)

    return (
        time[srt],
        flux[srt]
    )

# ============================================================
# BUILD GLOBAL LC
# ============================================================

def build_global_lc(sectors):

    all_time = []
    all_flux = []
    all_sector = []

    for sector, fname in sectors:

        t, f = load_sector_file(
            fname
        )

        all_time.append(t)
        all_flux.append(f)

        all_sector.append(
            np.full(
                len(t),
                sector
            )
        )

    time = np.concatenate(
        all_time
    )

    flux = np.concatenate(
        all_flux
    )

    sector_id = np.concatenate(
        all_sector
    )

    srt = np.argsort(time)

    return (
        time[srt],
        flux[srt],
        sector_id[srt]
    )

# ============================================================
# BLS
# ============================================================

def get_bls_period(
    time,
    flux,
    min_period=0.5,
    max_period=20
):

    flux = (
        flux /
        np.nanmedian(flux)
    )

    periods = np.linspace(
        min_period,
        max_period,
        20000
    )

    durations = np.linspace(
        0.02,
        0.20,
        25
    )

    bls = BoxLeastSquares(
        time,
        flux
    )

    result = bls.power(
        periods,
        durations
    )

    best = np.argmax(
        result.power
    )

    return (
        result.period[best],
        result.transit_time[best],
        result.depth[best],
        result.duration[best],
        result
    )

# ============================================================
# PHASE PLOT
# ============================================================

def phase_plot(
    ax,
    time,
    flux,
    sector_id,
    period,
    title
):

    flux = (
        flux /
        np.nanmedian(flux)
    )

    unique_sectors = np.unique(
        sector_id
    )

    for sector in unique_sectors:

        mask = (
            sector_id == sector
        )

        phase = (
            (time[mask] % period)
            / period
        )

        ax.scatter(
            phase,
            flux[mask],
            s=2,
            alpha=0.10,
            label=f"S{sector}"
        )

    # ----------------------------------------
    # BINNED CURVE
    # ----------------------------------------

    phase_all = (
        (time % period)
        / period
    )

    bins = np.linspace(
        0,
        1,
        120
    )

    centers = 0.5 * (
        bins[:-1]
        +
        bins[1:]
    )

    medians = []

    for i in range(
        len(bins)-1
    ):

        m = (
            (phase_all >= bins[i])
            &
            (phase_all < bins[i+1])
        )

        if np.sum(m) > 5:

            medians.append(
                np.nanmedian(
                    flux[m]
                )
            )

        else:

            medians.append(
                np.nan
            )

    ax.plot(
        centers,
        medians,
        color='k',
        lw=3
    )

    ax.set_xlim(
        0,
        1
    )

    ax.set_title(
        title
    )

    ax.set_xlabel(
        "Phase"
    )

# ============================================================
# PHASE DIAGNOSTIC
# ============================================================

def save_phase_diagnostic(
    time,
    flux,
    sector_id,
    P,
    outfile
):

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(18,5)
    )

    phase_plot(
        axes[0],
        time,
        flux,
        sector_id,
        P/2,
        f"P/2 = {P/2:.4f} d"
    )

    phase_plot(
        axes[1],
        time,
        flux,
        sector_id,
        P,
        f"P = {P:.4f} d"
    )

    phase_plot(
        axes[2],
        time,
        flux,
        sector_id,
        2*P,
        f"2P = {2*P:.4f} d"
    )

    axes[0].set_ylabel(
        "Normalized Flux"
    )

    axes[2].legend(
        fontsize=8,
        loc="best"
    )

    plt.tight_layout()

    plt.savefig(
        outfile,
        dpi=300
    )

    plt.close()

# ============================================================
# MAIN LOOP
# ============================================================

summary = []

for tic in sorted(
    tic_files.keys()
):

    print(
        "\n====================="
    )

    print(
        f"TIC {tic}"
    )

    try:

        sectors = sorted(
            tic_files[tic]
        )

        (
            time_all,
            flux_all,
            sector_id
        ) = build_global_lc(
            sectors
        )

        (
            P_bls,
            t0,
            depth,
            duration,
            result
        ) = get_bls_period(
            time_all,
            flux_all
        )

        print(
            f"P_BLS = {P_bls:.5f}"
        )

        summary.append({

            "tic": tic,
            "P_bls": P_bls,
            "depth": depth,
            "duration": duration,
            "n_points": len(time_all)

        })

        # ----------------------------------------
        # BLS
        # ----------------------------------------

        plt.figure(
            figsize=(7,4)
        )

        plt.plot(
            result.period,
            result.power,
            color='k'
        )

        plt.axvline(
            P_bls,
            color='red',
            lw=2
        )

        plt.xlabel(
            "Period [d]"
        )

        plt.ylabel(
            "BLS Power"
        )

        plt.title(
            f"TIC {tic}"
        )

        plt.tight_layout()

        plt.savefig(
            os.path.join(
                OUTPUT_DIR,
                f"TIC_{tic}_BLS.png"
            ),
            dpi=300
        )

        plt.close()

        # ----------------------------------------
        # PHASE CHECK
        # ----------------------------------------

        save_phase_diagnostic(
            time_all,
            flux_all,
            sector_id,
            P_bls,
            os.path.join(
                OUTPUT_DIR,
                f"TIC_{tic}_phasecheck.png"
            )
        )

    except Exception as e:

        print(
            f"ERROR: {e}"
        )

# ============================================================
# SAVE TABLE
# ============================================================

summary = pd.DataFrame(
    summary
)

summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "orbital_candidates.csv"
    ),
    index=False
)

print("\nDONE")

print(summary)

In [ ]:
def phase_bin_curve(
    phase,
    flux,
    nbins=100
):

    bins = np.linspace(
        0,
        1,
        nbins + 1
    )

    centers = 0.5 * (
        bins[:-1]
        + bins[1:]
    )

    medians = np.full(
        nbins,
        np.nan
    )

    for i in range(nbins):

        m = (
            (phase >= bins[i])
            &
            (phase < bins[i+1])
        )

        if np.sum(m) > 5:

            medians[i] = np.nanmedian(
                flux[m]
            )

    return (
        centers,
        medians
    )

In [ ]:
def find_primary_eclipse(
    phase,
    flux
):

    centers, medians = (
        phase_bin_curve(
            phase,
            flux,
            nbins=120
        )
    )

    idx = np.nanargmin(
        medians
    )

    phase_min = centers[idx]

    return (
        phase_min,
        centers,
        medians
    )

In [ ]:
def align_sector(
    phase,
    phase_min
):

    phase_aligned = (
        phase - phase_min
    ) % 1

    return phase_aligned

In [ ]:
def build_aligned_phase_curves(
    sectors,
    P_orb
):

    sector_curves = {}

    for sector, fname in sectors:

        time, flux = load_sector_file(
            fname
        )

        flux = (
            flux /
            np.nanmedian(flux)
        )

        phase = (
            (time % P_orb)
            / P_orb
        )

        (
            phase_min,
            centers,
            medians
        ) = find_primary_eclipse(
            phase,
            flux
        )

        phase_aligned = align_sector(
            phase,
            phase_min
        )

        (
            centers2,
            medians2
        ) = phase_bin_curve(
            phase_aligned,
            flux
        )

        sector_curves[sector] = {

            "phase":
            phase_aligned,

            "flux":
            flux,

            "phase_bin":
            centers2,

            "flux_bin":
            medians2,

            "phase_min":
            phase_min

        }

    return sector_curves

In [ ]:
def plot_aligned_phase_curve(
    sector_curves,
    tic,
    P_orb
):

    fig, ax = plt.subplots(
        figsize=(10,6)
    )

    all_phase = []
    all_flux = []

    for sector in sorted(
        sector_curves.keys()
    ):

        d = sector_curves[
            sector
        ]

        ax.scatter(
            d["phase"],
            d["flux"],
            s=2,
            alpha=0.10,
            label=f"S{sector}"
        )

        ax.plot(
            d["phase_bin"],
            d["flux_bin"],
            lw=3
        )

        all_phase.append(
            d["phase"]
        )

        all_flux.append(
            d["flux"]
        )

    all_phase = np.concatenate(
        all_phase
    )

    all_flux = np.concatenate(
        all_flux
    )

    phase_bin, flux_bin = (
        phase_bin_curve(
            all_phase,
            all_flux,
            nbins=120
        )
    )

    ax.plot(
        phase_bin,
        flux_bin,
        color='k',
        lw=4,
        label="Combined"
    )

    ax.set_xlim(0,1)

    ax.set_xlabel(
        "Phase"
    )

    ax.set_ylabel(
        "Normalized Flux"
    )

    ax.set_title(
        f"TIC {tic}\n"
        f"P = {P_orb:.5f} d"
    )

    ax.legend()

    plt.tight_layout()

    plt.show()

In [ ]:
tic = 11300044

P_orb = 2.1038

sectors = sorted(
    tic_files[tic]
)

sector_curves = (
    build_aligned_phase_curves(
        sectors,
        P_orb
    )
)

plot_aligned_phase_curve(
    sector_curves,
    tic,
    P_orb
)

In [ ]:
tic = 11300044

P_orb = 2.1038

sector = 6

sectors = sorted(
    tic_files[tic]
)

fname = dict(sectors)[sector]

print(fname)

time, flux = load_sector_file(
    fname
)

flux = flux / np.nanmedian(flux)

plt.figure(figsize=(12,4))

plt.scatter(
    time,
    flux,
    s=2
)

plt.xlabel("Time [BTJD]")
plt.ylabel("Normalized Flux")

plt.title(
    f"TIC {tic} - Sector {sector}"
)

plt.show()

In [ ]:
phase = (
    (time % P_orb)
    / P_orb
)

centers, medians = phase_bin_curve(
    phase,
    flux,
    nbins=120
)

plt.figure(figsize=(8,5))

plt.scatter(
    phase,
    flux,
    s=2,
    alpha=0.15
)

plt.plot(
    centers,
    medians,
    lw=3
)

plt.xlabel("Phase")
plt.ylabel("Flux")

plt.title(
    f"TIC {tic} - Sector {sector}"
)

plt.show()

In [ ]:
tic = 11300044

P_orb = 2.1038

sector = 98

sectors = sorted(
    tic_files[tic]
)

fname = dict(sectors)[sector]

print(fname)

time, flux = load_sector_file(
    fname
)

flux = flux / np.nanmedian(flux)

plt.figure(figsize=(12,4))

plt.scatter(
    time,
    flux,
    s=2
)

plt.xlabel("Time [BTJD]")
plt.ylabel("Normalized Flux")

plt.title(
    f"TIC {tic} - Sector {sector}"
)

plt.show()

In [ ]:
phase = (
    (time % P_orb)
    / P_orb
)

centers, medians = phase_bin_curve(
    phase,
    flux,
    nbins=120
)

plt.figure(figsize=(8,5))

plt.scatter(
    phase,
    flux,
    s=2,
    alpha=0.15
)

plt.plot(
    centers,
    medians,
    lw=3
)

plt.xlabel("Phase")
plt.ylabel("Flux")

plt.title(
    f"TIC {tic} - Sector {sector}"
)

plt.show()

In [ ]:
tic = 11300044

P_orb = 2.1038

sector = 32

sectors = sorted(
    tic_files[tic]
)

fname = dict(sectors)[sector]

print(fname)

time, flux = load_sector_file(
    fname
)

flux = flux / np.nanmedian(flux)

plt.figure(figsize=(12,4))

plt.scatter(
    time,
    flux,
    s=2
)

plt.xlabel("Time [BTJD]")
plt.ylabel("Normalized Flux")

plt.title(
    f"TIC {tic} - Sector {sector}"
)

plt.show()

In [ ]:
# ============================================================
# PHASE CURVES BY SECTOR
# (NO ALIGNMENT)
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# BINNING
# ============================================================

def phase_bin_curve(
    phase,
    flux,
    nbins=120
):

    bins = np.linspace(
        0,
        1,
        nbins + 1
    )

    centers = 0.5 * (
        bins[:-1]
        + bins[1:]
    )

    medians = np.full(
        nbins,
        np.nan
    )

    for i in range(nbins):

        mask = (
            (phase >= bins[i])
            &
            (phase < bins[i+1])
        )

        if np.sum(mask) > 5:

            medians[i] = np.nanmedian(
                flux[mask]
            )

    return (
        centers,
        medians
    )

# ============================================================
# BUILD SECTOR PHASE CURVES
# ============================================================

def build_sector_phase_curves(
    sectors,
    P_orb
):

    sector_curves = {}

    for sector, fname in sectors:

        time, flux = load_sector_file(
            fname
        )

        # ----------------------------------------
        # simple normalization
        # ----------------------------------------

        flux = (
            flux /
            np.nanmedian(flux)
        )

        # ----------------------------------------
        # phase fold
        # ----------------------------------------

        phase = (
            (time % P_orb)
            / P_orb
        )

        # ----------------------------------------
        # binned curve
        # ----------------------------------------

        phase_bin, flux_bin = (
            phase_bin_curve(
                phase,
                flux,
                nbins=120
            )
        )

        sector_curves[sector] = {

            "time": time,
            "flux": flux,

            "phase": phase,

            "phase_bin": phase_bin,
            "flux_bin": flux_bin

        }

    return sector_curves

# ============================================================
# PLOT PHASE CURVES
# ============================================================

def plot_phase_curve(
    sector_curves,
    tic,
    P_orb
):

    fig, ax = plt.subplots(
        figsize=(10,6)
    )

    all_phase = []
    all_flux = []

    # ----------------------------------------
    # individual sectors
    # ----------------------------------------

    for sector in sorted(
        sector_curves.keys()
    ):

        d = sector_curves[
            sector
        ]

        ax.scatter(
            d["phase"],
            d["flux"],
            s=2,
            alpha=0.08,
            label=f"S{sector}"
        )

        ax.plot(
            d["phase_bin"],
            d["flux_bin"],
            lw=3
        )

        all_phase.append(
            d["phase"]
        )

        all_flux.append(
            d["flux"]
        )

    # ----------------------------------------
    # combined curve
    # ----------------------------------------

    all_phase = np.concatenate(
        all_phase
    )

    all_flux = np.concatenate(
        all_flux
    )

    phase_bin_global, flux_bin_global = (
        phase_bin_curve(
            all_phase,
            all_flux,
            nbins=120
        )
    )

    ax.plot(
        phase_bin_global,
        flux_bin_global,
        color="k",
        lw=4,
        label="Combined"
    )

    # ----------------------------------------
    # cosmetics
    # ----------------------------------------

    ax.set_xlim(
        0,
        1
    )

    ax.set_xlabel(
        "Phase"
    )

    ax.set_ylabel(
        "Normalized Flux"
    )

    ax.set_title(
        f"TIC {tic}\n"
        f"P = {P_orb:.5f} d"
    )

    ax.legend()

    plt.tight_layout()

    plt.show()

# ============================================================
# PLOT INDIVIDUAL SECTOR
# ============================================================

def plot_single_sector(
    sector_curves,
    sector,
    tic,
    P_orb
):

    d = sector_curves[
        sector
    ]

    fig, ax = plt.subplots(
        figsize=(8,5)
    )

    ax.scatter(
        d["phase"],
        d["flux"],
        s=2,
        alpha=0.10
    )

    ax.plot(
        d["phase_bin"],
        d["flux_bin"],
        lw=4
    )

    ax.set_xlim(
        0,
        1
    )

    ax.set_xlabel(
        "Phase"
    )

    ax.set_ylabel(
        "Normalized Flux"
    )

    ax.set_title(
        f"TIC {tic} - Sector {sector}\n"
        f"P = {P_orb:.5f} d"
    )

    plt.tight_layout()

    plt.show()

# ============================================================
# EXAMPLE
# ============================================================

tic = 11300044

P_orb = 2.1038

sectors = sorted(
    tic_files[tic]
)

sector_curves = (
    build_sector_phase_curves(
        sectors,
        P_orb
    )
)

# combined figure

plot_phase_curve(
    sector_curves,
    tic,
    P_orb
)

# individual sectors

for sector in sorted(
    sector_curves.keys()
):

    plot_single_sector(
        sector_curves,
        sector,
        tic,
        P_orb
    )